# NILM XGBoost Pipeline (Classifier + Regressor)
**Kaggle setup:** Settings -> Accelerator -> GPU (T4 x2 or P100). Required for `device='cuda'` below.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score


## 1. Window shifter and data path

In [2]:
class WindowShifter:
    @staticmethod
    def shift(df, n):
        keep = df[['Time', 'Aggregate']]
        rest = df.drop(columns=['Time', 'Unix', 'Aggregate'])
        frames = [
            keep.shift(i).rename(columns={'Time': f'Time_t{i}', 'Aggregate': f'Aggregate_t{i}'})
            for i in range(n)
        ]
        return pd.concat(frames + [rest], axis=1).dropna()


data_path = '/kaggle/input/datasets/meowll/nilm-data/House2_full.csv'


## 2. Load, clean, window, and engineer features

In [3]:
df = pd.read_csv(data_path, encoding='utf-8')

app_cols = [f'Appliance{i}' for i in range(1, 10)]

# fix invalid readings where Aggregate < sum(appliances)
invalid_mask = df['Aggregate'] < df[app_cols].sum(axis=1)
df.loc[invalid_mask, app_cols] = np.nan
df[app_cols] = df[app_cols].interpolate(method='linear').bfill().ffill()
df.loc[invalid_mask, 'Aggregate'] = np.nan
df['Aggregate'] = df['Aggregate'].interpolate(method='linear').bfill().ffill()

df = WindowShifter.shift(df, 50)

aggregate_cols = [c for c in df.columns[:-9] if 'Time' not in c]
df['remain'] = (df['Aggregate_t0'] - df[app_cols].sum(axis=1)).clip(lower=0)

dt = pd.to_datetime(df['Time_t0'], format='%Y-%m-%d %H:%M:%S')
df['dow'] = dt.dt.day_of_week.astype('int8')
df['dom'] = dt.dt.day.astype('int8')
df['hour'] = dt.dt.hour.astype('int8')

# drop Time_t* columns (not used as features) and downcast to float32 to save RAM
df = df.drop(columns=[c for c in df.columns if c.startswith('Time_t')])
for c in aggregate_cols + ['remain'] + app_cols:
    df[c] = df[c].astype('float32')

df.head()


/tmp/ipykernel_58/2704847298.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['remain'] = (df['Aggregate_t0'] - df[app_cols].sum(axis=1)).clip(lower=0)
/tmp/ipykernel_58/2704847298.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['dow'] = dt.dt.day_of_week.astype('int8')
/tmp/ipykernel_58/2704847298.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) inst

,Aggregate_t0,Aggregate_t1,Aggregate_t2,Aggregate_t3,Aggregate_t4,Aggregate_t5,Aggregate_t6,Aggregate_t7,Aggregate_t8,Aggregate_t9,...,Appliance4,Appliance5,Appliance6,Appliance7,Appliance8,Appliance9,remain,dow,dom,hour
49,695.0,694.0,702.0,698.0,696.0,690.0,690.0,689.0,695.0,686.0,...,0.0,0.0,0.0,0.0,0.0,0.0,609.0,1,17,22
50,683.0,695.0,694.0,702.0,698.0,696.0,690.0,690.0,689.0,695.0,...,0.0,0.0,0.0,0.0,0.0,0.0,597.0,1,17,22
51,686.0,683.0,695.0,694.0,702.0,698.0,696.0,690.0,690.0,689.0,...,0.0,0.0,0.0,0.0,0.0,0.0,601.0,1,17,22
52,691.0,686.0,683.0,695.0,694.0,702.0,698.0,696.0,690.0,690.0,...,0.0,0.0,0.0,0.0,0.0,0.0,606.0,1,17,22
53,691.0,691.0,686.0,683.0,695.0,694.0,702.0,698.0,696.0,690.0,...,0.0,0.0,0.0,0.0,0.0,0.0,606.0,1,17,22


## 3. Feature/target columns and on/off thresholds

In [4]:
x_cols = ['dow', 'dom', 'hour'] + aggregate_cols
y_cols = app_cols + ['remain']

on_threshold = {
    'Appliance1': 15,   # Fridge-Freezer
    'Appliance2': 20,   # Washing Machine
    'Appliance3': 20,   # Dishwasher
    'Appliance4': 15,   # Television
    'Appliance5': 50,   # Microwave
    'Appliance6': 50,   # Toaster
    'Appliance7': 10,   # Hi-Fi
    'Appliance8': 100,  # Kettle
    'Appliance9': 5,    # Oven Extractor Fan
}

train_mask = ((df['dom'] - 1) // 7 + 1) % 2 == 0
test_mask = ~train_mask


## 4. Classifier: is appliance below the on/off threshold

In [5]:
y_class = pd.DataFrame({app: df[app] < on_threshold[app] for app in app_cols})

X_train_class, y_train_class = df.loc[train_mask, x_cols], y_class.loc[train_mask]
X_train_class, X_val_class, y_train_class, y_val_class = train_test_split(
    X_train_class, y_train_class, test_size=0.01
)
X_test_class, y_test_class = df.loc[test_mask, x_cols], y_class.loc[test_mask]


In [6]:
classifier = {}
for app in app_cols:
    clf = xgb.XGBClassifier(
        n_estimators=5000,
        learning_rate=0.05,
        early_stopping_rounds=50,
        tree_method='hist',
        device='cuda',
        eval_metric='logloss',
    )
    clf.fit(X_train_class, y_train_class[app], eval_set=[(X_val_class, y_val_class[app])], verbose=100)
    classifier[app] = clf


[0]	validation_0-logloss:0.65380
[100]	validation_0-logloss:0.36396
[200]	validation_0-logloss:0.33697
[300]	validation_0-logloss:0.31755
[400]	validation_0-logloss:0.30455
[500]	validation_0-logloss:0.29398
[600]	validation_0-logloss:0.28480
[700]	validation_0-logloss:0.27825
[800]	validation_0-logloss:0.27052
[900]	validation_0-logloss:0.26497
[1000]	validation_0-logloss:0.25908
[1100]	validation_0-logloss:0.25373
[1200]	validation_0-logloss:0.24982
[1300]	validation_0-logloss:0.24518
[1400]	validation_0-logloss:0.24079
[1500]	validation_0-logloss:0.23686
[1600]	validation_0-logloss:0.23368
[1700]	validation_0-logloss:0.23038
[1800]	validation_0-logloss:0.22743
[1900]	validation_0-logloss:0.22427
[2000]	validation_0-logloss:0.22202
[2100]	validation_0-logloss:0.21831
[2200]	validation_0-logloss:0.21655
[2300]	validation_0-logloss:0.21359
[2400]	validation_0-logloss:0.21131
[2500]	validation_0-logloss:0.20771
[2600]	validation_0-logloss:0.20568
[2700]	validation_0-logloss:0.20352
[280

In [7]:
clf_f1 = {app: f1_score(classifier[app].predict(X_test_class), y_test_class[app]) for app in app_cols}
pd.Series(clf_f1, name='F1 (on/off)')


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [20:23:42] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Appliance1    0.840209
Appliance2    0.983682
Appliance3    0.957432
Appliance4    0.944372
Appliance5    0.999012
Appliance6    0.999435
Appliance7    0.907283
Appliance8    0.998753
Appliance9    0.997417
Name: F1 (on/off), dtype: float64

## 5. Regressor: predict power draw per appliance

In [8]:
X_train, y_train = df.loc[train_mask, x_cols], df.loc[train_mask, y_cols]
X_test, y_test = df.loc[test_mask, x_cols][:70000], df.loc[test_mask, y_cols][:70000]
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.05)


In [9]:
regressor = {}
for app in app_cols:
    model = xgb.XGBRegressor(
        n_estimators=5000,
        learning_rate=0.05,
        early_stopping_rounds=50,
        tree_method='hist',
        device='cuda',
    )
    model.fit(X_train, y_train[app], eval_set=[(X_val, y_val[app])], verbose=100)
    regressor[app] = model


[0]	validation_0-rmse:42.53972
[100]	validation_0-rmse:31.18561
[200]	validation_0-rmse:29.82659
[300]	validation_0-rmse:29.04990
[400]	validation_0-rmse:28.42667
[500]	validation_0-rmse:27.94841
[600]	validation_0-rmse:27.52973
[700]	validation_0-rmse:27.03447
[800]	validation_0-rmse:26.79405
[900]	validation_0-rmse:26.54495
[1000]	validation_0-rmse:26.30208
[1100]	validation_0-rmse:25.99677
[1200]	validation_0-rmse:25.81214
[1300]	validation_0-rmse:25.60317
[1400]	validation_0-rmse:25.44133
[1500]	validation_0-rmse:25.23084
[1600]	validation_0-rmse:25.07256
[1700]	validation_0-rmse:24.95607
[1800]	validation_0-rmse:24.75033
[1900]	validation_0-rmse:24.59838
[2000]	validation_0-rmse:24.41173
[2100]	validation_0-rmse:24.21691
[2200]	validation_0-rmse:24.08415
[2300]	validation_0-rmse:23.95492
[2400]	validation_0-rmse:23.77556
[2500]	validation_0-rmse:23.63110
[2600]	validation_0-rmse:23.51521
[2700]	validation_0-rmse:23.37146
[2800]	validation_0-rmse:23.22554
[2900]	validation_0-rmse:2

## 6. Inference and energy-based evaluation

In [10]:
pred = [regressor[app].predict(X_test) for app in app_cols]
overall_pred = pd.DataFrame(pred).transpose()
overall_pred = overall_pred.clip(lower=0)


In [11]:
def calculate_nilm_metrics(y_true, y_pred, appliance_names):
    y_t, y_p = np.array(y_true), np.array(y_pred)
    eps = 1e-9
    sum_min = np.minimum(y_p, y_t).sum(axis=0)
    sum_pred, sum_true = y_p.sum(axis=0), y_t.sum(axis=0)

    precision = sum_min / (sum_pred + eps)
    recall = sum_min / (sum_true + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    abs_err = np.abs(y_t - y_p).sum(axis=0)
    nep = abs_err / (sum_true + eps)
    mae = abs_err / y_t.shape[0]

    out = pd.DataFrame({
        'Appliance': appliance_names,
        'Precision (PE)': precision.round(4),
        'Recall (RE)': recall.round(4),
        'F1-Score (FE)': f1.round(4),
        'NEP': nep.round(4),
        'MAE (W)': mae.round(4),
    })
    avg = ['--- AVERAGE ---'] + out.iloc[:, 1:].mean().round(4).tolist()
    out.loc[len(out)] = avg
    return out


metrics_table = calculate_nilm_metrics(
    y_test.reset_index(drop=True).drop(columns=['remain']), overall_pred, app_cols
)
print('XGBoost Regressor rating table:')
display(metrics_table)


XGBoost Regressor rating table:


,Appliance,Precision (PE),Recall (RE),F1-Score (FE),NEP,MAE (W)
0,Appliance1,0.6980,0.6571,0.6769,0.6272,21.416500
1,Appliance2,0.3900,0.4528,0.4190,1.2555,20.743999
2,Appliance3,0.7185,0.6619,0.6890,0.5974,38.756599
3,Appliance4,0.4072,0.2019,0.2699,1.0920,7.949500
4,Appliance5,0.4998,0.4910,0.4953,1.0004,3.561900
5,Appliance6,0.4271,0.5569,0.4835,1.1900,3.135100
6,Appliance7,0.0674,0.3149,0.1111,5.0399,3.496300
7,Appliance8,0.8776,0.8475,0.8623,0.2708,5.738300
8,Appliance9,0.0563,0.0638,0.0598,2.0071,0.414400
9,--- AVERAGE ---,0.4602,0.4720,0.4519,1.4534,11.690300
